# 06.1 - Perceptron & Activation Functions

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

The perceptron is the smallest unit of a neural network — a single neuron that computes a weighted sum of inputs, adds a bias, and passes the result through an activation function. Activation functions introduce nonlinearity so networks can learn complex mappings.

## 2. Why Does This Matter?

Every deep network is built from these atomic units. Without understanding the perceptron, layers become black boxes. Without understanding activations, you cannot diagnose training failures like dead neurons or vanishing gradients.

## 3. Prerequisites

- Phase 02 (linear algebra, calculus), Phase 05 (gradient-based optimization concepts)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain the perceptron's weighted-sum + activation computation
- Implement sigmoid, tanh, ReLU, Leaky ReLU, and softmax from scratch
- Train a single-neuron classifier on AND/OR gates
- Understand why a single perceptron cannot solve XOR

## 5. Mental Model

A perceptron is a decision machine: it takes in signals, weighs them by importance, adds a threshold, and decides how strongly to fire. The activation function controls the shape of that firing behavior.

```
z = X @ w + b   (weighted sum)
a = activation(z)  (firing behavior)
```


## 6. Visualize Backend

Use a non-blocking matplotlib backend so the notebook runs headless.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt

print("Backend ready:", matplotlib.get_backend())


Backend ready: Agg


## 7. Activation Functions from Scratch

Implement the core activations and plot them.


In [2]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def tanh(z):
    return np.tanh(z)

def relu(z):
    return np.maximum(0, z)

def leaky_relu(z, alpha=0.01):
    return np.where(z >= 0, z, alpha * z)

def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

z = np.linspace(-6, 6, 300)
plt.figure(figsize=(8, 5))
plt.plot(z, sigmoid(z), label="sigmoid", linewidth=2)
plt.plot(z, tanh(z), label="tanh", linewidth=2)
plt.plot(z, relu(z), label="ReLU", linewidth=2)
plt.plot(z, leaky_relu(z), label="Leaky ReLU", linewidth=2)
plt.axhline(0, color='gray', lw=0.5)
plt.ylim(-1.5, 2.0)
plt.xlabel("z")
plt.ylabel("activation(z)")
plt.title("Common Activation Functions")
plt.legend()
plt.savefig('_tmp_act.png', dpi=80)
plt.close()

# Softmax sanity check: outputs sum to 1 across classes
logits = np.array([[2.0, 1.0, 0.1], [1.0, 3.0, -1.0]])
sm = softmax(logits)
print("Softmax rows sum:", sm.sum(axis=1))
print("Softmax output:\n", np.round(sm, 3))


Softmax rows sum: [1. 1.]
Softmax output:
 [[0.659 0.242 0.099]
 [0.117 0.867 0.016]]


## 8. Perceptron from Scratch

Train a single neuron on the AND gate using binary cross-entropy gradient descent.


In [3]:
# Perceptron forward + training (binary classification via sigmoid)
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([0, 0, 0, 1], dtype=float)  # AND gate

rng = np.random.default_rng(42)
w = rng.normal(0, 0.3, size=X.shape[1])
b = 0.0

def perceptron_predict(X, w, b):
    return sigmoid(X @ w + b)

def bce(y_true, y_pred):
    eps = 1e-8
    return -np.mean(y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps))

lr = 1.0
epochs = 300
losses = []
for epoch in range(epochs):
    pred = perceptron_predict(X, w, b)
    err = pred - y
    w -= lr * (X.T @ err) / len(y)
    b -= lr * err.mean()
    losses.append(bce(y, pred))

print(f"Final loss: {losses[-1]:.4f}")
print("Predictions (AND):", perceptron_predict(X, w, b).round(2))
print("Learned weights:", np.round(w, 2), "bias:", round(b, 2))


Final loss: 0.0566
Predictions (AND): [0.   0.06 0.06 0.91]
Learned weights: [5.01 5.01] bias: -7.7


## 9. Why a Single Perceptron Fails on XOR

AND and OR are linearly separable; XOR is not. A single neuron draws one straight decision boundary and cannot separate the XOR pattern.


In [4]:
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

rng = np.random.default_rng(7)
w = rng.normal(0, 0.3, size=2)
b = 0.0
for epoch in range(2000):
    pred = perceptron_predict(X, w, b)
    err = pred - y_xor
    w -= lr * (X.T @ err) / len(y)
    b -= lr * err.mean()

print("XOR predictions (single perceptron):", perceptron_predict(X, w, b).round(2))
print("Best achievable with one line            :", np.array([0,1,1,0]))
print("\nA single neuron cannot separate XOR — it is not linearly separable.")
print("We need at least one hidden layer (Unit 06.4) to bend the boundary.")


XOR predictions (single perceptron):

 [0.5 0.5 0.5 0.5]
Best achievable with one line            : [0 1 1 0]

A single neuron cannot separate XOR — it is not linearly separable.
We need at least one hidden layer (Unit 06.4) to bend the boundary.


## 10. Dead Neurons: ReLU vs Leaky ReLU

ReLU outputs exactly 0 for all negative inputs, so a neuron can become permanently 'dead'. Leaky ReLU keeps a small gradient flowing.


In [5]:
neg = np.linspace(-10, 0, 1000)
dead_relu = np.sum(relu(neg) == 0)
dead_leaky = np.sum(leaky_relu(neg, 0.01) == 0)
print(f"ReLU:      {dead_relu}/1000 negative inputs produce exactly 0 (dead)")
print(f"Leaky ReLU:{dead_leaky}/1000 negative inputs produce exactly 0")
print("\nLeaky ReLU never fully dies — it always has a small negative slope.")


ReLU:      1000/1000 negative inputs produce exactly 0 (dead)
Leaky ReLU:1/1000 negative inputs produce exactly 0

Leaky ReLU never fully dies — it always has a small negative slope.


## 11. Debugging / Common Errors

**Debugging table (key rows):**

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Output stuck near 0.5 | Sigmoid saturating | Print pre-activation `z` | Use ReLU hidden layers, check weight scale |
| Loss is NaN | Sigmoid overflow | Check `z` values | Clip inputs or use stable log-sum-exp |
| No learning | All weights zero | Print weights | Random initialization |
| Dead neurons | ReLU always 0 | Count zero activations | Leaky ReLU or lower LR |

**Common mistakes:** using sigmoid in hidden layers, forgetting to clip sigmoid, using softmax for binary output, all-zero init.

## 12. Real-World Considerations

- Word-frequency features feed a spam-filter perceptron; each weight reflects how indicative a word is.
- Softmax turns logits into a valid probability distribution for multi-class output.

## 13. When NOT to Use

- A single perceptron for any non-linearly-separable problem.
- Sigmoid/tanh in deep hidden stacks (vanishing gradients).

## 14. Challenge

Implement a softmax with a stable log-sum-exp trick and verify it handles very large logits without overflow.


In [6]:
# Challenge: numerically stable softmax with large logits
def softmax_stable(z):
    z = z - np.max(z, axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

big = np.array([[1000.0, 1000.0, 1000.0], [2000.0, 0.0, -2000.0]])
print("Stable softmax rows sum:", softmax_stable(big).sum(axis=1))
print("Output:\n", np.round(softmax_stable(big), 4))
print("\nSubtracting the max prevents exp overflow even for large logits.")


Stable softmax rows sum: [1. 1.]
Output:
 [[0.3333 0.3333 0.3333]
 [1.     0.     0.    ]]

Subtracting the max prevents exp overflow even for large logits.


## 15. Closed-Book Recall

Without looking back:

1. Why can a single perceptron not solve XOR?
2. What happens to sigmoid gradients at very large or very small inputs?
3. Why is ReLU preferred over sigmoid in hidden layers?
4. What does the bias term allow a neuron to do that weights alone cannot?

## 16. Teach-Back Questions

Explain to another person:

- How a perceptron computes and learns.
- The trade-off between ReLU and Leaky ReLU.

## 17. Summary

You implemented activations from scratch, trained a from-scratch perceptron on AND, demonstrated why XOR needs a hidden layer, and inspected dead-neuron behavior.

## 18. Further Experiment

- Train the same perceptron on OR and NAND gates.
- Plot the decision boundary of the trained perceptron.

## 19. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
